In [16]:
import pandas as pd
import requests
from datetime import datetime

def get_cbr_key_rate():
    # Ключевая ставка ЦБ РФ была введена 13.09.2013
    start_date = "13.09.2013"
    # Текущая дата
    end_date = datetime.now().strftime("%d.%m.%Y")

    # Ссылка на страницу базы данных ЦБ РФ
    url = f"https://cbr.ru/hd_base/KeyRate/?UniDbQuery.Posted=True&UniDbQuery.From={start_date}&UniDbQuery.To={end_date}"

    # ЦБ РФ активно блокирует автоматические запросы, поэтому обязательно нужен User-Agent
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    print(f"Скачиваем данные с {start_date} по {end_date}...")
    response = requests.get(url, headers=headers)
    
    # Проверяем успешность запроса
    if response.status_code != 200:
        raise Exception(f"Ошибка при обращении к сайту ЦБ РФ. Код: {response.status_code}")

    # Ищем все таблицы на HTML-странице
    tables = pd.read_html(response.text)
    
    # Нужная нам таблица со ставками — первая на странице
    df = tables[0]

    # Переименовываем колонки для удобства
    df.columns = ["Дата", "Ключевая_ставка"]

    # --- Очистка и форматирование данных ---
    
    # Преобразуем строку с датой в объект datetime
    df["Дата"] = pd.to_datetime(df["Дата"], format="%d.%m.%Y")
    
    # Заменяем запятые на точки в числах (особенность русской локали) и переводим в числа с плавающей точкой
    df["Ключевая_ставка"] = df["Ключевая_ставка"].astype(str).str.replace(',', '.').astype(float) / 100.0

    # Сортируем данные от старых к новым
    df = df.sort_values(by="Дата").reset_index(drop=True)

    return df, response

if __name__ == "__main__":
    try:
        # Получаем датафрейм
        rate_df, resp = get_cbr_key_rate()
        
        # Выводим последние 5 изменений
        print("\nПоследние значения ключевой ставки:")
        print(rate_df.tail())
        
        # Сохраняем результат в Excel или CSV
        #filename = "cbr_key_rate.csv"
        filename = "cbr_key_rate.xlsx"
        # rate_df.to_csv(filename, index=False, encoding='utf-8')
        rate_df.to_excel(filename, index=False)
        print(f"\nДанные успешно сохранены в файл {filename}")
        
    except Exception as e:
        print(f"Произошла ошибка: {e}")

Скачиваем данные с 13.09.2013 по 07.04.2026...


C:\Users\Ershov\AppData\Local\Temp\ipykernel_19840\2873468520.py:27: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)



Последние значения ключевой ставки:
           Дата  Ключевая_ставка
3140 2026-04-01             15.0
3141 2026-04-02             15.0
3142 2026-04-03             15.0
3143 2026-04-06             15.0
3144 2026-04-07             15.0

Данные успешно сохранены в файл cbr_key_rate.xlsx


In [8]:
resp.text

'\r\n\r\n<!DOCTYPE html>\r\n<html>\r\n<head>\r\n    \r\n\r\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />\r\n<meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">\r\n<meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no">\r\n<meta name="format-detection" content="telephone=no" />\r\n<meta name="zoom:lang" content="ru" />\r\n    <meta name="zoom:last-modified" content="Mon, 06 Apr 2026 21:00:00 GMT" />\r\n    <meta name="zoom:tags" content="БазыДанных, InMaterials" />\r\n<title>Ключевая ставка Банка России | Банк России</title>\r\n\r\n\r\n\r\n    <meta property="og:image" content="/common/images/share-1.jpg" />\r\n\r\n    \r\n\r\n\r\n            <link rel="stylesheet" type="text/css" href="/common/libs/jquery-ui/jquery-ui.min.css?v=v478607061" media="all">\r\n\r\n            <!--[if IE 9]><link rel="stylesheet" type="text/css" href="/common/style/main-ie9.css?v=v4139323626" media="all"><![endif]-->\r\n\r\n            <!--[i

In [9]:
tables = pd.read_html(resp.text)

C:\Users\Ershov\AppData\Local\Temp\ipykernel_19840\2924359120.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text)


In [12]:
tables[0]

,Дата,Ставка
0,07.04.2026,1500
1,06.04.2026,1500
2,03.04.2026,1500
3,02.04.2026,1500
4,01.04.2026,1500
...,...,...
3140,23.09.2013,550
3141,20.09.2013,550
3142,19.09.2013,550
3143,18.09.2013,550
